# House Image Discovery with Cost Tracking

Finds the best panoramic views of **one target house** and describes it:

1. query pano metadata within `RADIUS_M` of the house (parameterised, dry-run first);
2. for every pano, check every ground camera in code and keep the one that can show the whole
   house (its width, base and roof line) without black border; rank the views by distance and
   viewing angle, at most `MAX_PER_SEQ` per drive;
3. render a perspective view framed on the house and ask Gemini for a `HouseView`
   (visibility, box, occlusion, stories, material, roof type);
4. triangulate the house from the boxes of at least two panos and derive the house id from
   that point. With fewer than two usable views the house is reported as unlocated. The id is
   run-local (a 2 m cell of the triangulated point, which box noise can move); to keep ids
   across runs, pass the previous run's locations to `views.match_house_ids`.

**Data model.** Each pano is a rosette of 7 frames (`observation_id` suffix `_0`..`_6`):
cameras 0-5 look horizontally ~60 deg apart, camera 6 looks at the sky. Frames are portrait
3648x5472 wide-angle images with a per-camera `camera_pose` (position, heading, pitch, roll)
and no published intrinsics; `svi_geo` ships a calibrated fisheye model for them. Panos are
chained into drive sequences from `snapshot_id` + capture time + distance, which is used here
to spread the chosen views over different drives.

**Division of labour.** All geometry is code: camera choice, view framing and rendering with
the calibrated lens model, box -> bearing conversion and triangulation. Gemini only answers
perception questions about views that code has rendered, and every answer is validated against
a Pydantic schema and range-checked.

**Terms of use.** The frames come from your Imagery Insights dataset and are governed by the
Imagery Insights terms of your Google Cloud agreement.

- Do not redistribute the frames, or crops rendered from them, outside your organisation.
- Frames are downloaded with your own credentials. They are not cached on disk unless you pass
  `cache_dir=` to `GcsImageFetcher`. Cached frames expire after 24 hours
  (`images.DEFAULT_CACHE_TTL_S`), and `fetcher.purge_expired()` deletes stale files.
- Keep the attribution: every figure that shows imagery is labelled "Imagery © Google", and
  every map carries the same credit next to the base-map attribution.

In [ ]:
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
# On Colab this installs only svi_geo (and folium if missing): its lower bounds are the
# versions preinstalled on Colab (runtime 2026.07 and current), so nothing is upgraded.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF
# (a commit SHA gives reproducible runs).
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "svi-gemini-improvements")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

**Authentication and permissions.**

- **Colab:** the next cell calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** Application Default Credentials, set up once with
  `gcloud auth application-default login`.
- **Optional workstation helpers** (only if plain ADC fails, e.g. Vertex AI returns 401 on
  some managed workstations): `SVI_USE_GCLOUD_TOKEN=1` mints short-lived user tokens with
  `gcloud auth print-access-token`, and `SVI_ECP_PROXY_URL=http://localhost:<port>` routes
  Vertex AI calls through a local enterprise-certificate (mTLS) proxy.
- **Parameters:** set `PROJECT_ID` (your billing project) and `GCS_BUCKET` (the frame bucket
  linked to your Imagery Insights dataset) in the parameters cell or as environment
  variables. The notebook stops if either is missing; nothing is guessed.
- **IAM roles you need** (this notebook changes no IAM settings): `roles/bigquery.jobUser` on
  `PROJECT_ID`, `roles/bigquery.dataViewer` on the linked dataset,
  `roles/storage.objectViewer` on the frame bucket and `roles/aiplatform.user` for Gemini.

In [ ]:
# Colab: authenticate the notebook user (Application Default Credentials elsewhere).
try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

In [ ]:
import os

# ---- Parameters -------------------------------------------------------------------------
PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")  # billing project (required)
DATASET_ID = "imagery_insights___us"  # Imagery Insights dataset linked into PROJECT_ID
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")  # frame bucket linked to DATASET_ID (required)
MODEL = "gemini-3.5-flash"
LOCATION = "global"
MAX_GEMINI_CALLS = 500  # hard cap on Gemini requests for this notebook; None = unlimited
CONCURRENCY = 16  # parallel Gemini requests

LAT, LNG = 28.05047, -81.96015  # target house (roughly the facade centre)
RADIUS_M = 80.0  # pano search radius
MAX_VIEWS = 12  # Gemini calls
MAX_PER_SEQ = 4  # at most this many views from one drive sequence
HOUSE_WIDTH_M = 14.0  # the view must fit about this width at the house
HOUSE_HEIGHT_M = 8.0  # and the house from its base up to about this height
VIEW_W, VIEW_H = 1024, 768

In [ ]:
from google.cloud import storage

from svi_geo import attribution, auth, config, data, images, rosette
from svi_geo import gemini_client as gc

# Fails fast (ConfigError) if PROJECT_ID or GCS_BUCKET is missing; no silent fallbacks.
settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
print("project:", PROJECT_ID, "| frames bucket:", BUCKET)

creds = auth.get_credentials()  # None -> Application Default Credentials
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)  # pano_observations_latest
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),  # pano views only
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(gc.make_vertex_client(PROJECT_ID, LOCATION, creds), model=MODEL),
    max_calls=MAX_GEMINI_CALLS,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()  # self-calibrated 7-camera rosette model shipped with svi_geo
print("intrinsics fitted:", intr.fitted)

In [ ]:
import numpy as np

from svi_geo import entities as ent
from svi_geo import schemas, sequence, usecases, views

VARIANT = usecases.DEFAULT_VARIANT

In [ ]:
# Metadata only (no gcs_uri column: selecting it scans ~1.9 GB); the frame URI is derived
# from snapshot_id + observation_id. Parameterised SQL, dry-run first, 2 GB cap.
PANO_META_SQL = data.pano_meta_sql(TABLE)
params = data.pano_meta_params(lat=LAT, lng=LNG, radius_m=RADIUS_M)
print(f"dry run: {bq.dry_run(PANO_META_SQL, params) / 1e9:.2f} GB")
frames = data.normalize_frames(bq.run(PANO_META_SQL, params))
frames["gcs_uri"] = [
    data.gcs_uri_for(BUCKET, s, o)
    for s, o in zip(frames["snapshot_id"], frames["observation_id"], strict=True)
]
panos = sequence.build_sequences(data.panos_from_frames(frames))
frames = frames.merge(panos[["pano_id", "seq_id"]], on="pano_id", how="left")
print(f"{len(frames)} frames, {len(panos)} panos, {panos.seq_id.nunique()} drive sequences")

## Choose and rank candidate views in code

In [ ]:
cands = views.house_view_candidates(
    frames,
    LAT,
    LNG,
    intr,
    aspect=VIEW_W / VIEW_H,
    house_width_m=HOUSE_WIDTH_M,
    house_height_m=HOUSE_HEIGHT_M,
    max_dist_m=RADIUS_M,
)
ranked = views.rank_house_views(cands, max_per_seq=MAX_PER_SEQ, n=MAX_VIEWS)
print(
    f"{len(cands)} panos in radius, {int(cands.cam_k.notna().sum())} can show the whole house "
    f"-> {len(ranked)} views selected"
)
if ranked.empty:
    raise RuntimeError(
        "No pano within RADIUS_M has a camera that shows the whole house without black "
        "border (too close, too far, or no coverage). Move LAT/LNG to the facade, increase "
        "RADIUS_M, or reduce HOUSE_WIDTH_M / HOUSE_HEIGHT_M."
    )
ranked[["pano_id", "seq_id", "cam_k", "dist_m", "off_axis_deg", "hfov", "pitch", "score"]]

In [ ]:
def house_view(row):
    img = images.decode(fetcher.fetch(row["gcs_uri"]))
    view = views.view_for(row, VIEW_W, VIEW_H)
    return rosette.render_perspective(img, intr, row["camera_pose"], view, int(row["cam_k"]))


cand_rows = ranked.to_dict("records")
crops = [house_view(r) for r in cand_rows]
black = [
    rosette.view_black_fraction(
        intr, r["camera_pose"], views.view_for(r, VIEW_W, VIEW_H), int(r["cam_k"])
    )
    for r in cand_rows
]
# black_fraction is analytic sensor coverage only; dark_pixel_max is measured on the sent
# images and also counts near-black content in the frames (e.g. redaction blobs).
print(f"black_fraction_max={max(black):.4f}")
print(f"dark_pixel_max={max(images.dark_pixel_fraction(c) for c in crops):.4f}")
assert max(black) < 0.01, "a rendered view extends past the sensor"
PROMPT = (
    "The image is centred on the bearing of a target house. Is a house visible at the CENTRE? "
    "If yes, box it ([ymin, xmin, ymax, xmax], 0-1000), and report occlusion, the visible "
    "fraction of its facade, stories, exterior material and roof type, with your confidence."
)
print(f"planned: {len(crops)} calls, est. ${gc.estimate_cost(len(crops)):.2f}")
# The API enforces the HouseView schema (JSON mode); code rejects degenerate or out-of-range
# boxes and re-asks once.
views_out = await runner.ask_many(
    [([PROMPT, c], schemas.HouseView) for c in crops],
    validator=lambda v: v.validate_in_image(VIEW_W, VIEW_H),
)
print(runner.cost.summary())
runner.check()  # raises if any request failed, with the first error messages
assert any(v is not None and v.house_visible for v in views_out), "no view shows the house"

In [ ]:
res = ranked.assign(
    visible=[v.house_visible if v else None for v in views_out],
    occlusion=[v.occlusion.value if v else None for v in views_out],
    facade_fraction=[v.facade_visible_fraction if v else None for v in views_out],
    confidence=[v.confidence if v else None for v in views_out],
)
# a box far from the centre means the model looked at a different building
res["centred"] = [
    bool(v and v.house_visible and v.box_2d and abs((v.box_2d[1] + v.box_2d[3]) / 2 - 500) < 250)
    for v in views_out
]
sightings = [
    views.HouseSighting(
        r["pano_id"],
        r["camera_pose"],
        views.view_for(r, VIEW_W, VIEW_H),
        schemas.box_2d_to_pixels(v.box_2d, VIEW_W, VIEW_H),
    )
    for r, v, c in zip(cand_rows, views_out, res.centred, strict=True)
    if c
]
print(f"centred_views={int(res.centred.sum())}")
location, status = views.triangulate_house(sightings)
if location is None:
    print(f"house {status}: fewer than 2 panos gave a consistent centred box")
else:
    print(
        f"house {status} from {location.n_views} panos at ({location.lat:.6f}, "
        f"{location.lng:.6f}), rms {location.rms_m:.2f} m; house_id: {location.entity_id}"
    )


def value(x):
    return getattr(x, "value", x)


ok = [v for v, c in zip(views_out, res.centred, strict=True) if c]
attrs = {
    k: ent.fuse_attribute(((value(getattr(v, k)), v.confidence) for v in ok), ignore={"UNKNOWN"})
    for k in ("stories", "exterior_material", "roof_type")
}
print("fused attributes (value, weight share):", attrs)
res[["pano_id", "cam_k", "dist_m", "visible", "centred", "occlusion", "facade_fraction"]]

In [ ]:
import matplotlib.pyplot as plt

best = (
    res[res.visible.fillna(False) & res.centred]
    .sort_values(["facade_fraction", "confidence"], ascending=False)
    .head(4)
)
if best.empty:
    print("No view shows the house centred; nothing to plot.")
else:
    fig, axes = plt.subplots(1, len(best), figsize=(5 * len(best), 4))
    for ax, (i, r) in zip(np.atleast_1d(axes), best.iterrows(), strict=True):
        ax.imshow(crops[i][..., ::-1])
        attribution.add_to_axes(ax)
        ax.set_title(f"{r.dist_m:.0f} m, facade {r.facade_fraction:.2f}")
        ax.axis("off")
    plt.show()
print(runner.cost.summary())